# 📋 LOG05 | 미션 10 | Student Workbook

---

## 이번 미션에서 하는 일

**무인 보급장비 투입 전 상태·책임·중지조건 표준 체크리스트 완성**

1. 실습 목적 및 개요 역할 및 배경: 학습자는 무인 보급장비 투입 전 상태·책임·중지조건 체크리스트 교육용 실습의 참여자로서 임무를 수행합니다. 핵심 과제: 미션 1부터 미션 9까지의 모든 산출물을 종합하여 무인 보급장비 투입 전 상태·책임·중지조건 표준 체크리스트(표준 지침서)를 완성하고, 모의 심의를 거쳐 최종 제출합니다. 2. 주의 및 검증 지침 데이터 해석의 엄정성: 자료나 AI 설명을 철저히 검증하지 않으면 미확인 기록이나 모의 회신을 실제 현장 확인이나 승인으로 오인할 수 있으므로 주의해야 합니다. 수행 방식: 본인 판단, AI 초안 및 역할 대화, 근거 대조 및 보완, 최종 기록의 전 과정을 원본 자료 범위 내에서 엄정하게 수행합니다. 3. 산출물 및 소요 시간 안내 결과 파일 목록: manual/log05_standard_operational_manual.md LOG05_M10_forms.md forms/LOG05_M10_forms.md 시간 안내: 원본 예상 시간은 3시간이며, CMS 편성은 원본 durationHours 에 맞춰 3시간으로 구성되어 있습니다 (실제 소요 시간은 실습 완료 후 확인).

**학습 목표:** 가상 자료를 읽고 자신의 최초 판단을 적은 다음, AI 초안을 원자료와 대조하여 문서와 검증 근거를 완성합니다.

실제 장비 제어·비행·파괴·무력화 방법을 설계하거나 실행하는 훈련은 아닙니다.

**준비물:** 이 Notebook, 미션 10의 가상 자료 및 빈 작성 양식, 기관이 허용한 AI 환경(가능한 경우).

**학습 환경:** Colab은 인터넷이 필요합니다.

폐쇄망에서는 사전 설치된 Jupyter 또는 별도 Markdown 양식으로 진행할 수 있으며 외부 AI를 사용할 수 없으면 미실시로 기록하세요.

**학습 순서:** 자료 먼저 읽기 → 본인 생각 기록 → AI 질문 → 원문 근거 대조 → 수정 → 저장·제출.

**구분:** 파일 저장 / 실습 수행 / 평가 기준 충족 / 실제 교관 승인은 각각 다른 상태입니다.

## 📦 실습자료 자동 준비 (모든 미션 공통)

- 아래 **「LOG05 실습자료 자동 준비」** 코드셀을 **먼저 실행**하세요.
- 실습자료(미션 lesson·forms, 공통 안내)가 노트북에 내장되어 있어 ZIP 업로드 없이 `LOG05_실습자료` 폴더에 자동으로 풀립니다.
- 화면에서 **「압축 해제 완료」** 메시지와 실습자료 폴더 경로를 확인한 다음, 아래 미션 실습을 진행하세요.

> **주의:** 자료 준비만으로 실습 결과나 교관 승인이 완료되는 것은 아닙니다. 아래 코드셀의 소스코드는 기본 숨김으로 설정되어 있습니다.

In [ ]:
# @title 📦 LOG05 실습자료 자동 준비 (ZIP 업로드 불필요) { display-mode: "form" }
# 학습자 화면에는 결과와 업로드 인터페이스만 표시하도록 구성합니다.
from pathlib import Path
import os, zipfile, stat
from IPython.display import display, Markdown

LOG05_PROJECT_DIR = Path.cwd() / "LOG05_실습자료"
LOG05_PROJECT_READY = False

def _log05_extract_zip(zip_path):
    global LOG05_PROJECT_READY
    zip_path = Path(zip_path)
    if not zipfile.is_zipfile(zip_path):
        raise ValueError("유효한 ZIP 압축파일이 아닙니다. 실습자료 ZIP을 다시 선택해 주세요.")
    dst = LOG05_PROJECT_DIR.resolve()
    dst.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        entries = archive.infolist()
        if len(entries) > 20000 or sum(info.file_size for info in entries) > 1024**3:
            raise ValueError("압축파일의 항목 수 또는 해제 후 용량이 허용 범위를 초과합니다.")
        for info in entries:
            item = Path(info.filename.replace('\\', '/'))
            if item.is_absolute() or '..' in item.parts or not info.filename.strip():
                raise ValueError("안전하지 않은 ZIP 내부 경로가 포함되어 있습니다: " + info.filename)
            target = (dst / item).resolve()
            if not target.is_relative_to(dst):
                raise ValueError("실습 폴더 밖으로 벗어나는 파일 경로입니다.")
            mode = (info.external_attr >> 16) & 0o170000
            if mode == stat.S_IFLNK:
                raise ValueError("심볼릭 링크가 포함된 ZIP은 허용하지 않습니다.")
        for info in entries:
            target = dst / Path(info.filename.replace('\\', '/'))
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(info) as src, target.open('wb') as out:
                    import shutil
                    shutil.copyfileobj(src, out)
    files = [p for p in dst.rglob('*') if p.is_file()]
    if not files:
        raise ValueError("ZIP에 압축 해제할 실습자료 파일이 없습니다.")
    LOG05_PROJECT_READY = True
    print("✅ LOG05 실습자료 압축 해제 완료:", dst)
    print("📄 확인된 파일 수:", len(files))
    print("ℹ️ 실습자료 폴더:", dst)
    print("※ 업로드·압축 해제는 학습 수행 및 평가 완료를 뜻하지 않습니다.")
    return dst

# 실습자료가 노트북 안에 내장되어 있어 ZIP 업로드 없이 자동으로 준비됩니다.
# 내장 파일 5개: LOG05_Colab_실행안내.md, LOG05_M10_forms.md, LOG05_M10_lesson.md, LOG05_공통_AI역할실습안내.md, LOG05_공통_작성안내.md
import base64 as _embedded_b64
_EMBEDDED_ZIP_B64 = (
    'UEsDBBQAAAgIAAAAIVzAhdWSSQcAAMMOAAAbAAAATE9HMDVfQ29sYWJf7Iuk7ZaJ7JWI64K0Lm1kbVddbxNXEH3Pr7hSxSMmDtDAI2orQCJq1RaQ+hIZSEtK'
    'qFECRfShWiebaBMbYkOcbJK12RTnq3XbTbyOHcm85Kf0ce/d/9Azc+/aC1gCFtl3556ZOefM+DNx6+vro5eF2i7FhaY8KQhVbMTry0JVHTkfjoxkM/rE5PP8'
    '7KN7+fyjzPSTF7/ck7s98UV+JndPbZSV7Qm1EUYtX6iGJU9toRYtfG5eREC10o7XbLnjKb8Wr3Qzv00/UTUcc5ty770sOrLYyAjl2+o4VL6n6hXZ6oofbn4T'
    'tXoiagOWHVcP++f6dwhVb2q4qmaJqBuonRCfVZR9JOLVUL5cI5jy5T9qPVRvArWPN6orAGNCjYxlRNRqx0ttce0mUvg7rvomcbVpiVtTc3P5Xxho7T1lB1Sq'
    '1hVqvgmsIi6VZfHgrKNqrmx2zzrXblLOcn8n6uJMrUcvyoOe8nGrXxjcijC2XLX5yuB9Uj8Uycdja00d/33WkcW2bFl89bserpbBdirAtW9uini+IVfWRFy1'
    '8Q6wL6W+j9rlKLQEgiDSv11Zd4U6DeUbD2DNPfVd1FDOu0KeWLJECKuyFVJEnYaqL0p/N666nxbtYkZ8+9V3t299L5C2qpfRepRAqPWKKtYY8hEwBVSwudyv'
    'U5OzU3PPZp7ykXc9ZfORuEqNxr+Gbun2bnaBScSvumq/gp6CNUKubqKggrhx4ono+D1RCQEJgO1Rk3XXEZAqK8v9aHEV51utpMrmlMtc1aWIOr3oKOBCxZtV'
    '9Bf/NX1ue+qPRaFsP15g1HL+IDotJHW4lNH55aYnH0znZvI/PZsirEM7CX2VCSyCREEBf0CCEGxN5x06oB5dDvLZeP55QCnGWyVV9NG4fScJ8LEg7j58ceFG'
    '/vmFuw9zXOaoZcdbL8860bEVL/n0RDN61Nsf87OP53R7dSH6MVA2ioszYEO8CUbt28qzSQaa59w/t0WFizpduWdrXXTVXm0oSy5nOGfoWLmOKnK7SKENi0XN'
    'RRfxawd5kXihiSjwiHUG0LdZ+vh61hxVG03qrKqvybfLV8bORYFlAl8dPWcCM6KTQ7VzNAyRkEuv+aY3R1F3GQX1y8pxSbamxvz6fFPWS5yraf7KKSXp9bi1'
    'QzxEqKUSK7Belu9KDAtaxPlDcHsTomqj45Zc2VW1kPSmGhWOsWnjODGXSOdXh9bw84yYGIeMG2orxP0QMLkeIdHOOjE6PqkcD8yO3apWuHHbzMOnj2c0ZUPp'
    'N8g+qlS9MucJ4XRLEg61FcCd+g7ed+LYteGsxjPkfkl7QnRcIo+WxWaatXyGXBTtg25jOxgDOwVnGxrH5Hoe/9V3TN0uPLm1Wr7abkLxn+UOjgqCV9/9z9rk'
    'XBxb7vUJghvibTtlppq4JhujHvkn4p6edQYdNqWQFYjglPX0gSlQsWulJHUg05kNMh4Z56YMMBKnNEzBeD6eNkSFQGhV01kcUksvBTtbjSdG2lV08ULHCBZJ'
    'Ks9nVtUdxGUNGFkVvSiwh1CfDHLBMDaueDg3XA5UAR4D4vaX57Nnndt3zo9xpxYK8QLnDixyr5lUpmThG+iGJYgn3KGOfBPb5BEMZygxQWVlp3+RaUWrTeOO'
    'gKZnEN/ouNrj/Cpw0uu6OkNxG8aR7FYaqB97VnrIDarFn+81aXx/2A2qyYbLTak6ifeRwDHZtw6HavEKEoFVAd7+coKB+n3j+4lbnN6Oh+w/FKs8tuOVnvEV'
    'Djzw274uyZL4Xar4k9n8z1P32cVplhVPKQW1iNT2amyYvNWwgzPlAJWNxpgVz0f6Nl5F2w5Sc/BxbvbRZO7+02e5mcmn04+nmLW0QZltj8hUIKn3Ky//pU4w'
    'H40YMU8Ag96kpA3qC6ab8YJFmj/o0sgwAxSRMbkTqg8pw8hViOkKsh0bhUIFDXzMroqXiPjESfYbdk0KTY8NdHAJVtYQRCDvIDVWCeN8kDaobRz0EH+dODvf'
    'jI5Cika+TDPlBDakFhwQxo5oBL1mPmp6DOYmNcH3YAWUIqmZGcNEw2ZgD583E+OIdVm5BSyWvBNh4NTSm4BZDiEuPQB4KYErn7omJy2sVDLkcqwmroQp/Ooy'
    'gMDax2B2w/aDkewowFylPO5kf78zzhkaD4G9+DbaCwA0lpPFp97Uvdb7Yndn6Nqhly+QDpPBGhvFl/31UsIxK95F/RENePci6j40JX2UANGaS7PmksjGr1yC'
    'pr1Ab4LM9MD8SEiW/GTntAZv0J6oSZPM4pTlpRF/4kTJ533nxBgcagXZrN4ric0YQoWQ28pbopi4dNaZ+Bx/r3C2LEMoOTuKIUK9Y9d13I9WXs4GxedtRBOE'
    'CldQfkUkW5zxHK06rIe6bgNtJQulNuuBVS0mA/7Zk5l87sHcBd4d+j/BDH/TayhiQliOqS7xoo+KZzDclBIODmAKukDUN/rRtHXIU6jYgECAWJNME9SMJwx9'
    '/ePELFZ69cPx1Oon3wbkJnXbgOF1FH3nWZnY5qAd/wNQSwMEFAAAAAgAAAAhXBZJjMihCAAAyBkAABIAAABMT0cwNV9NMTBfZm9ybXMubWTtWF1vG8cVfeev'
    'GMAoIqGSKFmW6+pNL0kNJEiBPOShKBQBTpsCthXYLooAQkBKK4MxWZhEuOLSXgrLRhIll0ZociVRqPyin5K87cz+h5x7Z2Z3RYpOW6Tui2HD5s7O17333HPP'
    '3Wviw48/mF8SHy3MC7VbU84roXZqqtzK5dSxr/6xLdRmT534snu+LD67t3b/r2t383fX/zy/tPrw0dr9O2sP7qyuf/n5g7VHf1m/v3Z3Vc+Yu3fns1xu1uyo'
    'dqsXp7J3hF1/LNRVuycHjsiLqOer1vDXK7d/LLjL6eTWOSb3HRU4PFjeU4GfX7kt5ItD1fJE/PSbqFfAlqpZENGwp/YKwk7P3f384cP1+9ipKr+ryP1zoVqv'
    'o0Fgto7dI1kuyfLenFDPq3IwxEGDUDUdEdd82jvqNVUL222G6tkR/YofH2NR7Hqq0RWx6+OkkuxUkv1C+dQRsppsK78f4rbR4Bg74+ewrQL34pTmlc+EajxW'
    'rYpqObhVqPyADIjdw+hfHk48jM6KZpdc7tq1axjy5pKIOD58JxbErIhOz6NXPbowbBcqqEb9Ah6mVKegzg4xUcC/sftSdmgvs8F0LrdhxsVG4jS2UqhndeW+'
    'pmHeeYruve1dnJKTt4rGldN4b49sdOVJQWxgx9nZWXHFv3izMLuAJ/sXA9dHBxZHB26MDiyNDtwcHfjN6MCtywO5iZ68Dk8aPKlyl/+D/wKYu9klJ8mep7wi'
    'LCbHIeaxN8SWqvEydgP6cWnyhpluPZo8K99RZ96U2t2/OIX3p9/ktNR1GQuM78ZGFsdGboyNLI2N3BwZmeyeRQIasjNwRRQW5EF32T6q784JBFNCiOkZ4NiL'
    '+l2hSp4MQqF6r6PjLqEauS77rp1ll77aVrvAfqeIX1MaXtOjc1qhDPYJobLdMv7TU+iuwGrIftZJa/jibJjL3cJ6PO9Hg3MBdGIxp1mjF/UBeo8oAOGW3wYG'
    '/KpRtZlk8a6BDtwfV5Xf5swIfOWWMIIMewwO2SrGWz6TSlBUT45tzn9A5sY7FGPORqG8kir7F6dL8sSh8/CCl7J1Bw6o4cW2DM7Ibaoc0JXxIt75BvNgMFwk'
    '+3X8opNkpcBpm/LWZlfuVhiwT87AnlizVwMcBQyFA2kzMiBwaPOzJl53CrEPBzigMiG7Q0RIPqmLhYvTm/DXjIgrlSgMcacueT5u1tns5pAmLWGGIFfJWju5'
    'QKMnFuJnFRMxHJDFQEobHGx6zMSVYgLb4ZysRUzwhtbZ/YCcsY7JeOAQyRrvt/2ruFJT2xUkZytJQm8j4abxsXhjStx0cTxlumYHndBYpS+rb2fu9G9l9Vgy'
    '/l8GKIdQSeGb+NnfLbCTIqizLi3bMB7Bqy6LH3batEyWj+WAAhLq0uPwCzOqGiGAKeLnNi+xzcptgCfqV4TstKMh/NXYBsyIIHQB5PrO5VIk2Zct6PD38yrS'
    'l7AIfAPcNKFEkAYNq5M6oLXMUd/s0nV01KhM00B5j8EJzFMcdbgtCLwq7YkdgDnCVOw6qIO80uAid2UEJ/5PZaJSoPSRL46oNmxYYFN97b9kwdFFuiLfoHAs'
    'tSWzrBcnzIJW4XQAEo/qYLKfKb8TQTDyhEeNAbKcIpwUMMM7Pt0oic3AgRbkJABbWZKGMKhUZfnQ8BjCSlZB3XSQfcRCRBLAjNz3Rivj/yZzcjAEZYq4azF6'
    'FfKltorEPdbWVK5p4GvnE8sYAahhwmSD8JT9BFmePAJ6OruEnlQlamuN9WMFgrFhKhZjgWFNsMyEV6tQJE8agssn+xa7cK7R5U4Qb7UI43QsCWx2rYHOfwsQ'
    'ogida6yvs6WPM411s2FrQTrzxVGStQmabbKiAAKseG+v/vuVTz7Jv79y+8M88TBbmAd8qETIkxJtaZ0xUnZ/IcC/0XImI7Q6xoXZLoSC0aqY4qH8c4aG4xGp'
    'JbYmVUPLQt3RVMk1LAuWmRCt6tTGpQ2O0HUPmW/bGztFk6z1ln7KoolHdEGi5spUU30T7Xe0Vyz1bOtE/Nt0mA45bmy5iatLwsHYOUccr/ohsykUSjEEnUMh'
    'MnWWPHBAmkrY4g968R+5mQIa0o5Ed1F0KFUV8UOhPj5HhS1hVRxY4mlGxi3Mz//KLL5Ky5man0q6SS0MQqv6HuWnbmXinRJMwaSZN/UCLFp0O6APJ1XZcy4L'
    'YvZpeQ/V7UpNzFKuVlWN5MD3dMSYNLMCjunEaLxQGMJJIcJVjumWaWqi0mPRVu/J/SGC8x6zlZZ+dBGtwKzxFLpPv/gq/7v1v+U//WLt0bIpNoJ2IpVgKxSX'
    'tySXdYXIMqzVEViRJSjynyWxDFpGmCLJDN04M9apDGaoge5OJpw4xi1GKlqeoO46NJkgq/DRHkfr2x5zs/sky8iUDE1fPm3SlwjVcVh9YyN10Epk0IaZQhXO'
    'pNMozbErtGzXxZy5KhFUP9Mj4zd9ddGnLHBa3OJe0ULfs1SviUjLZ4o8jFjG2q8/+u04/diA5bU/U6LNNjA6229ZfW8bJZ+roqWxgxYVQUp2o2SyUk03xWQq'
    'g0VrvrzqlFi/IJzoEdDdYAIDJa+VAH/AMV9D2Glqd9sWY423zNeVHRqXR6EwgkOTFss1sAc1Ed5YaAUrEce4hWz90/qDew/z/JFrFe5e5ee5e3c4ZbV7Tcn8'
    'Dz5uzYjxDWcmH5VxcaI9NuiP6e5TGFxnGEA4UxetWc7QGxUNLUsoD6xqYySo0j/5E1umE233kJiXWlBuLdPek7FABReWIw5aZbM79ecjdF3tjPp4B4y3DozF'
    'S8BYZGBcqlO8ulNUQY3ZuFWzeAAxyoMu6RcXFYl63RoVDl3W+MOIjlTbT4R+mLS4WqpbUuOCgYZ7GGab9bR4WUJ+B5C3D5AblwBygwGi78Wftq1AyXy2InVr'
    'QZItNfSJWgs/VgkI/qSvPCxnEk1gSi8CRUWTj4B03X8Hh7cPh58AUEsDBBQAAAAIAAAAIVwjmqdqMQoAAEIfAAATAAAATE9HMDVfTTEwX2xlc3Nvbi5tZO1Z'
    'W28bxxV+568YwCgsAbpRNyt6KfSkGqhQoC95CApFgNsmgG0VjouigGAs5ZVAi0xF1aK1lJfKMtaFSmlkRa1EEpBf9FOat53Z/9DvnJkllxcpcQC7LwbikJqd'
    'PdfvfOfM8I74/R8WJ2bEUnpC/NfaFbIWqHJTyPMgbL5QB0eyZYsoW1MHG0J5tlDPM9Fz97qhzjbUgY3Pwx11YqmKH54FItpx1aElVD2IMjV5XFNbh9FWU6iS'
    'reyzVOrOHSF/bKoNh6WUTlMpmWupnKfKlgh9C4tQ4d5TWUdtViBY5i1jjrZG+o4qOyJs+rJSFvISUstCGyLU3tuo6MFKmcvK3OFYrCn97DN6R6376tKVtStV'
    'hjubF1HxNCoGH8DZoXgZ+1pVZbvD8ujKRCAqOuG5J+QPVTYpV8MHPQ3PfNVyyepUalSo1wV53oSy51mx+PVTEf2riXfFlMq5oW//ZO2q3KFqVkL/36ocCFW0'
    'Zc75ySrSmxeu+n6j4+u8+PLRyuO/rzwcf7j614mZ5W+erjx+sPLkwfLq3/78ZOXp16uPVx4u6x1jjx58CQnh+QWCI9TBDqlUxaxcD1TJEgv3TYDNGlktq1fK'
    'g6Pld2rrwkSdc6wdoFRCHGQV5Js8CYml2q7KleEHRSrnRkVXqP1AORlKAwIlKM1OhlyCtDuEjLFuYSItRjsBFnOhDxEHRyK8qEFBeH7FaPKayqmJsO5jAcFI'
    'pdZ40xrJkj+c4ku0XYuKVUE+7tPfA94Sa3htdHRUDPg/nqTxHbKUlyGt+CBBsr6rXLv9ZCRe1/jiSDD84weABZxBZLfk1m77WXhRAASpFlolCGfgB1Z4ToKX'
    '0qx9kpzZP1WvAAOvSDvgw7mtNr/F636BP+EsBe24RpljTGPniTWCYF+EDXwiW2H9VET7eeiCuF1Vf4vVAxu1IVSjqjbzrPG6sTTFWqdYK4csKjlh/R277trR'
    'nn4d9bRGqQBYRbTVQkZGxCynSO9B3cojByrWa7JidbyZpowkas+DXRani79ExbeIJzCDN3Oe/N6X27ZYTD9bnNWx8qswp0jaw4aLmJHgSRj9GQufIUFcvdcN'
    '7bkwZbxmqhwBsaLSrkSAVd2hgMlcFdGkEBaBTKoN6cOXKF/AE9jjmmRMQcs9/JtjTbMkkfmC44IAcEDUSQarLF0Tln5/JN6BqIDX4CsHJnptcwbXa6RgGsJn'
    'WPg9Cm0AlIKUotdZ6VXxuf0CwG0/MACClYhLk3jNOA44bkivRRqJqKSXxRehpQAOb6qymZfHyP3Cwh9J6+wz49Ecl4aLMJAKigzc2fMpv2ZZvbmC8ZBcBh6A'
    'tKwjPYDHfwcYjLQZmiNAmwLpUR0uzZn0pG4s9EkUehddtmmoFciXLte0XoCJdSvaxJOTrKw1e+s2RZDUZRDv78IV5yqZdgqpKl+RwwUHdYm/f8vAO98h4MEJ'
    '9A+2hAsQKeQdpEjWi1w7saK7OsWsQmcC9MIFa3ERHOQhrbvgwf4nVuSiXdjgY7VXEHLXl+Wru2QWHJff1eR6xyzTuPL5MAiA7xoFGFgW6rsMvKQ2EQcHaTu3'
    'ZcU1ziZ8W2N+3OHSiy2PHBuyoZ5odnICGGUWgTk6eOw/scZBtm2Ljqos5pEF2mkwaFosdawTC7arFqCSF3LrmKIHLm8zoLHGgAJbx7o7B7N/2LhC16QeDODF'
    '1uy4Q52ugPKm1J5QXzIChhkvvE54QbmhURv2J+YqvqNlljykx4frRjyXMB6HKSZG5V5NXlo/1x5GqUHE/xFj9y5M9S5M9y7M9C7M9i7c612Y6164OZIDygvx'
    '8zJMQQiS7sTwOMWwzkYOU1m7gro2r5ntnX5q/gbvI9tDQBJgXfGHbwtaV2ftiV3fylTfynTfykzfymzPys3hmSKgJYlvvpvwxJAQYngw7VGbl/4puCDeFb+a'
    'ZMMhDa/h3j2aIs2gq+OntxhbacBtg55RnSQzQqluhf0sGA9LxEYAMOTTRAwQyJ1CPKa+4ppZ0xo1peH5Sy8uGV1CXNy3FQsNibpaBkjUbAyyYrIEMJLU04dE'
    'tlGD0czQSWjxhI1Raq9Pyy+iXkTNkacoaczV6J+ah02b56PJzxLzUROK79L0BtLj0ccSaRPkWc2gzJE91t3M2xW/7RbNdZ0JkXy/gcsxUEBBtxEztygf0MF5'
    'NoUM6l4DMG0MGpBNEzp9nMKJiCYjmjxfFxAw+0xuHVEG1UvfuKEQ8S4RKX1yAO7r2fYhMXkqoHPPpaunWaEud7k/24eYafigMBjW5GPd7xy7OhjugFfE4NWo'
    'BYiRbrfCPQHailmsmIDznBiDrX3mEYtU6NErYjddhcrJ4mBz3Zihhgl9dJbjEZP8O8Y5Uk9iFFxMFAxOjBOvXmAf8Axy0CcIPmTlLa6201hZjFrHDI7xodQA'
    'mw8/N+LVIIMwgnmekDkyCH1wu9SkTQQfwSDfqbQN2PNFGqcFgxwoSCKlwwHmyNzFaJpqNvjU1/FIY0ePsRx+kK3xjsoPMzNPXDr6gLtcr4atTPLEGRVLyAdI'
    'z5XbJU2R+jvaCN1pzHHviXHhxAdlWbcpUJr/oAryUqml9DMMpfoAiHd+bGoqhZ9s57jul+OEGizk3CQs9MF3Lo5aDD83NrRt1iSbtXCfu4TGpSFVOuYxLvli'
    'gIGnBaRSKvsfYPS6kcAbH2G6gMYA6iCMbTOzLCiFmG5rl4tQj0eaQLSLgwyd0ndCyRmBPUP38nYYneWd2L6bKJ2JnMuRK4xTaGAdGCKnQ1mAU6F2QFMfpbkZ'
    'JCHSIXMC8aU9yOBpNlgnmK9RYgAmitcFk8RGJ6GBs1SR6saUCYy5CetMhzEkYkKGU9SxWYUFsQnzDLqZX0s2nWw0DXXue0SsmtmIursNHqVyJ27IBHoI5wuC'
    'rAOu6BQQRHyhX/6T6L18MlddprELjk3fHhWUO3co0t9OEGh6YuI3fZdX7z8JCKLuutM9EPCkP3Lb/NnX9X9lz8JoQ7OBUfh/mgnYEM19sfOUus+/+uf471b/'
    'Mf75VytP58F1Tb7BgKT9b9v4ArDAywiqaVhcPGKKbi9gKuLM5YM04Q1zBWh70XPOYFzaCbS0UaspbuG+yQGznSY1VEK+QDGWl1kYylxkSo7DYki6PYls8sGO'
    '4yoLiNEhZ6vT7nsuCOPqpIzruqWLkfj6xlRGeozvpPXz9+XxefGheDw2Ux2XiaWpIA07JW8u9GGJ8sIJ1eeKcX03wSGn+7achw2czHFCkr7CAynKXMtMton7'
    'bI0JeZI3wvQUJk8D0g9CMMTCdG4uVp2+8JvLjfh+E77+ZfXJo2/G+SeAZYR7mf8ee/SAy0qHFxMCuod4j6vkEdEvcORmVYkQt2E6mcz+r2iX8+IjtctPePgo'
    'eJhK4uGXTyXzNx40P/RU8gkXHwUX00lcvN/wNy8+/PD3CQUfBQWpxI+mt/xmOm5qlX86bf9yOsYjRyWgX9kAD7puuQCp6GtMvYhUmUyhHbgezXsd5f8DUEsD'
    'BBQAAAgIAAAAIVwFXnF7IAYAAMkMAAAkAAAATE9HMDVf6rO17Ya1X0FJ7Jet7ZWg7Iuk7Iq17JWI64K0Lm1kbVZNb9tGEL3rVyyQa+smaYO0vuUUtGiaAuk9'
    'ORU9NCjQ9gdQNmXQplpLsGhTDilQiWTJqYLSEmVTqJyDf4qP3OV/6JvZ5YeSnAhR3J2ZN++9mTvi+6eP7z4Q2WKZ7y3Fo2+FOnmXe5FQ7kgdLIXyHLmTNBp3'
    '7gh5aSs7vLnK5la+hw9Oe2r+TijfUW7QaNxagfJsOfDNhfLfVLV8oQYd+aat+pbANULtzFRki7zdke5UhbaQcS/f9VSwlsNAqMRBOH59aOOIGvlb5gL6O+97'
    'Kkxlx5cHPb7JHSGZZUcFQ3V6LrI0ViNLqAh5ODdX6rKXxZY5pMJE5J6NlHMvoExwgOJkyxmqyj0/W0QmkpBzJ/dTOV7jqqY6TmpZqJMO4Nnj+IOxXNlC2Rfy'
    'YIxobpDFTTwDWw1sPO1Avm3R+1F+vC9U36YicE5O2vJopCbAY+Jsicfy6CJL97PFGvmdZ/Nz3I+ULAqPzOTZTCf3GcOnS1ShL6Rjy/Aa9+821SWXouL3wF8n'
    'uwzU61aJMkPPWHhRNm9v3VphoyGnaxXRqz66jOrofzVx5CylXIpAuDbyCnhAiCgwbWfw0lgOQ6QtXUe6oy2QhwDSYXHQLuqYDOk+lKy8A0SrPjedEF9mF+BG'
    'mFA1FPNVx+ShJk0VdQWQRp9NHnKRmo4ylvQiAb4msZsr3I1umMZTu7OrdXYRcyY702zVNPE1p99OCc6C86eJ8puNxufiyf1ttLiL9gIbBpz7xGUvllkcUDJp'
    'TFVSzVncJ5yRHvAq0NOk1l2UnYDq1dLhVOJXNSjkIc6vqy+4GxpvediSZ9ciP9zHS6IhETP0IS+kAFCZjHVcKfuvtnEX/zOMoRSRJZaKOiYjjQcSs9TwAgrC'
    'taOuWvki/zvNj8esv5WNNtZai4qBplAXLU3viZct8JRzDyJB0mBCRAcJdWbMR+2mtB5QWlSbpo6QbiLd1c0VnEe5UflTHawQLLtso9yyNcxNFmAUoAWVWnvQ'
    'HAPtLmsJlzLtGOHdWoSZPnxrAe5EhmvmE51LlMcKLI9zClV8poC7lAuL9WRYE/rZfEY3nccfKGlnJgft2tkPJGkXzCaL2YDo4bbIfXjITKhVIo8CcNzoSo3+'
    'KumixVsp4T5YRgJij53/U4tTiJSlk12lIn+F9r5tyWglCNABmRdI8rqlHL8oLD9tox+gBh8mFAobxdm2lfd7dag0T/UZo13+ELIcJhv2gBS7QBHFW+okpsSr'
    'ZAspSTBtN8yP6Yw8TzhndlcbQ6EIls2vYQ/sNodwyGktCppMQ8igR0rDizPA+Wb9SU5+vS1k5ODbvG1xRldT5V2Tb1BCiPVg05wKafiUXNUB1G24RiFx1TAG'
    'qwhQ2aWifBl7QNrxIQ/oZpHIsc+3BlExajbS+mabkMi7ATeMoGMBbFgloVWZ4Sb9jXnoKzR+nH0v5kmbDlmm57WI9+5uYx52gBXrWs4S5AsLxLyu8RidixgH'
    '46rkJ2xZm43WF5QHKzfD+7EetbO0UDrjh93B7det+clDoZwAgzT3vWIbAQUwshuNF7xjPH9y9+Hz8hvd8ef6y61f/nz56wuueBio/2ZCpm3oXZ3GEA4jcHJd'
    'YRWgTnqIn+6R3rTDY9FgzfAP7oJmXYkmVopugB+yU5GP22EYjfIAVbmZQE7lIFIDCHBsVAln2pABXH7MDmXzmD9rlYmGSaVeNGlTrMQtN/qCPjreZ7/CiGDZ'
    'CpAl32UxFHbAfkeTMmwbqbAtm8KresBd49V6w2H3W5AU2XJZ4vXvJw6NpMk+cfO7Z09/IPihYBYFop0m+E1a05X//NvvL/+g3dCwg4jD9qIXrRhzpcbRhu4+'
    '0/BDvMqmWAZbs3JlqzZRlo4UXSOzhsNwMZBNM5Q7QVmdzdQ87JcK+XiQGbNNHHCBRwybWU3LtGUAs47PZrhI9QGznv746Nkzhi7uZe/Tjy8vmqu7ysrQkwzk'
    'A3uLMcY7zheF557soY+gDLOz7im8OaLbGKuw8IDXDMtAoy2WN7ujT2xnNbCNaRug+XBBfSSmNxMujPddlrkeMJqSZCN6WR50CdpqJd8QPZriMJnjaSnJiKca'
    'PcwoIk1s8Khkx/9QSwMEFAAACAgAAAAhXA1xTZ/1CgAAohcAABwAAABMT0cwNV/qs7XthrVf7J6R7ISx7JWI64K0Lm1kfVjLbhtHFt3zKwowsslIsp5+LA1M'
    'oDEwwgyyCZDBwEhmlESIIwWSjCCAMWhKLYEm6ZiKSYuUmkzTlizRQ8MtsklRMLXxpyS7rup/mHNvVTeLsj0LPdjsqrqPc889t66Jv/5tcXpBRN1evNsTqrGn'
    '3DOhKjm5FWYyn34q26GqD4TshtHgkWocywtXxLm2auwI5btCbWfjbe/duTrbUQ0Xf4/21ImjmkF0FgrVCeNsW75sq/xRnB9gs3xZyPZAuVgh+65y63G1gq1k'
    'IScLR1MiChxsiI09cVPlqmq3if1k0RHaCm2EDKqqXqWXYLOIBoFs1lUdptQvo64v7txVNefq0e/Oo/NhdBYInCqPqzj91Slvsv86rvgCR8S1Mp7igK4PL+DC'
    'xSnMlMdDE5K40kqsVIUjslAHg96Iem35pBZXqliH0OVVvpe8K/sONseO/DLM6LWjgOIE7y6qdFKWQlfHusOS7A7IqKhXUl5THbTSeLg40IpTJnPtmlAnbvzs'
    'kVA5D3ZmMjMUvay4v7yxsbbKrjVK8kWRHdCB+WZt/YcNJLKnttp00n5JxLUKwhpX4MxWG25NCMRFHoeUJzyJK68nKJ77pahTFPKkGQ0G2G1I0ZanQ+XDaj9r'
    'e1voya5jfS3DI+UNZSsUvztV/S1iHspGUcSHrgnt706NlsR7nip4ozhnZqdEXAll4ULINwO1k7iEoNVDwh5Sr45wRsfFJzw9LCEFZHrUCaLuEE/6ZcQN7+WQ'
    'GIoE+0K4lQU4V6WoJMk8KKvOayzp/Nf2ohuqZmiZNDdFe1B0XwyRUjHHMIc521mG4GEpsZFzOUBZuWorwL44Jp+gFcYR5FAlFqrkSYPLYytE5uUTbOb5jCfs'
    '7vmcRz9LFscVF9Zy0gAi4IIOBoz8PeOFZe/8FPtQc22nYErBS3PPUdDW4iihztx4u64B6le4JFAwjPJOgCd2HfQ89XxHxMUiMME+oTKrgKvnAtu0RG6dRhfZ'
    'xJiFKWEyp/PJWDep0rnEcW8G2rDE0noxedvKXGrC0sw0njNc0/K5FQVcnGwRyk+zA8gN62zreQFqy4e/Ymn+3fnSDfzc4tD7LgiEi6IOHglxBpEBHZOrouqs'
    'CGe++O7nP5zyKN3xweN353fuchQarknfH05lQvxl7Se8aQCXUE7CSwwHDjeypXmBEsqw4dVffPfVJi3XQU/iZs6NiyVZOMVHOwNYRkmInzwCj1ie37mLz8Bb'
    'UT4F/msuEb5mPjwjXlV9jyoJVYszqILwEkBNq1QFGajZPDSCtDZCM1gbRJvJTMIBXz4PEFMK3uIMIYxIDaUTBRUDvsUb1mNVubRM5d5EO2JjxJaiKV+6RJf4'
    'Wh7D8bIrm57y6yB54n1+HMo3SLsnGM8pLpK1Luf1wMYCyGlxztQxs5EDW9+dL87iZ26Cv56nr2EwvmZXgJfFBfzcIH7DWfLVzjh7TQq5+yvtZDKcRoLbkw9K'
    'dVDKrajTStx5mfbBz2c0g5flb4/ErdlPhI6e2UrttxmkBsDVHA7m9/ueZOjzstvTnxiGZJf7LdU8My1Kvnr7sXaWLgl1qos6iAxDzqzpSPrQUUeaNC6BxC2L'
    'EDh8oG2pcv1jYBHVewo8VST17sCR+WMTcG02EGgCqmlAlj7QVNVuEUyX7KkZl8G5v2u6DZifHug9TCuzdQZYedCMgl+JmD30TfxlyYIKMrjSJNQosdMc9qTT'
    '992xLCdi6MUQ+Gc6hiH5C6xOyNMwOfcj2yomawGFI6jatqqWgeluTGLdPe707KdZxpjpulRYBD6bZykToCmc/vQMconJKGF/h2KB7stwYSOZZ0E+Bf9D+oXy'
    '2nChvthM1alCUVlFaGkWRHT38QRSWsJf0FrnkiuQPZ9gUoH5tLiSY92IQNEXgEF0Ti8cAj0tER8UYbrxcSI9+/wUKecWyPuNV66WnKYuSb8Qq7GlhopA5Wid'
    'Qnpvx5zl18FXz8L3iA306qiXdVnyUsAzPCgbOoBGOZoWlnTh/QAfxtBBbePVDiGhp4JL0p070r8wjUx32YLPioZTzDzLfQbrKLFjtKA34YJs0BNbv3D1siMc'
    'WZIHUWfb1JJ6GnxAmp5kIZG5FWTz0SA0Wn7U6xJb2Ub2r4GsV3lDNjSVbB8Bjh4WisUopMJqh6hW9DyKJboT0SCaMraGbgAZh9xJOrp+3CPqpB8S1EYRqt+y'
    'BPMUg8gq2wXPvWoUOlpR5BBabEkYIm7a9VlEXMlSEkKS3b6bdjHHPGClXuOZgLaGWTAa28hyMEqNVuTNACVJGIECQWn96c7dTCYpEUfM/+cGImreMpTKgNLH'
    '6zaNKhGz1EvqWoX4DAPLWL017RdXakTKjRKnkKcpkY5hiTTTvrjCtFrd+k0sGiVEx8/Jp15MI8rYSMSVctK0Ak8sYt7ABID8ebxxgj/k9BD7jbzQb30Ufck6'
    'Q0chkTeBDkQtX+mWCNnHUpX7uj5ayx5WNZXT6G3VqA9WK1pEkWxDfAwPA63PA5CBcfm9TpDkBxHbzonFlU0R/4Lx9IwWz97k3uBatLx0k3e9CBEzSqKOLxLW'
    '19JkK5wQS7dHU27S52en8QaIeF7MxL9USWdQO5ijZejYqCLmNi35El0+Ggz0VnTWE8BH20RmjPVtc5L+mrvGGHUkYkGjLqAGDx90R9eu1DzMsGY9CxPaw8xd'
    'aOgYcMzJFOAhhPW41tLKl3CBnVtBat2gySOYzttHwKCrR1toxhWdqExmaYFBkR9Q71jQKpAFFAOd1Ma8uuAmtkCyX54cxZW9RJKdNNS5a2EO6TPDwohd0etH'
    '3ImOiQ+EJiMgL5IomWG/m9x48HTTd2GVriz0qkIdX5tyNGA2SKH/7NZvJTehM32QsU4LOJYnuj+mimdM6jwLTfs0QkqQMOzzoEvn0KzmC26nePl9Bh35hmIp'
    '1M3AQkeMximmSz6JOnglZ/NVOqZdFUVi6RbXzzS6OY/EmDaTaxojK3OyPsTwTGoVeoBsONkjBKES9AgSdXuQfkkaUJswDS8gJuM0oLbrfDnzgQ5koyrp4shd'
    'rZXJPDQ0Ix4mF1567BYP8dXk5KQwv/HJrNT3OOKhdUX1f2+oJugs/Gu9Pxf1Alpy4pBRRd7vH+ldj35PJ/ifOJmN39+RjwPSBnmg9LBIN1r9LOeAh7RMhuej'
    'ez+trX//9dra91MrP/68+jXjDL0BkTloWeSl5aLY2Hzw7+XVzesba/cfbK6srZr5nYesPpi7rVnYZeBoySlzrqxfpsmrjYQmX8Gpy7JBeXK3R+MvfGwP9MUH'
    '8cCO4SaS2my8+PLu33WXc0zN0QZ04QIif+KSbtNMkvY5PqjcIrzFVVfPpt772m3ppsG22cbcqkj/SAaHeJW8YkEDhwdFAFEdBGQE6x5C99iADz7HmyPa5t2Y'
    's1mzoqeaiWkUaB1QoyGtihljar6vImjqjkk3gyXoFkuVvzdrMCQQk07OXL0Y7a1bFusONiWTsafitKGl/YzZkVRMfWirIL6awX5EByT56F2zmGsfWyDsAEg6'
    'Il5BA3upmSptQ7JAZEMRN345Gob2rMMoQu9lq9v4rWd5FQwQXnONmS9vrD1Y/9fyvbX1lW9XVr+6v3H925XNe+vL3yyvL6/i+ez07I2Z6enblMZGzhYaCNU2'
    'ZqPTzz+78+elz7Cx9mLlhx/X1jeJOZ+e0QXG1Szq9OhCuK5viWB9MoBUaTDlW6LRJVHdmZmmbno8vHpLlF4beenNF5FfcjOjFY9Ls8te6WqLyPwPUEsBAhQD'
    'FAAACAgAAAAhXMCF1ZJJBwAAww4AABsAAAAAAAAAAAAAAKQBAAAAAExPRzA1X0NvbGFiX+yLpO2WieyViOuCtC5tZFBLAQIUAxQAAAAIAAAAIVwWSYzIoQgA'
    'AMgZAAASAAAAAAAAAAAAAACkAYIHAABMT0cwNV9NMTBfZm9ybXMubWRQSwECFAMUAAAACAAAACFcI5qnajEKAABCHwAAEwAAAAAAAAAAAAAApAFTEAAATE9H'
    'MDVfTTEwX2xlc3Nvbi5tZFBLAQIUAxQAAAgIAAAAIVwFXnF7IAYAAMkMAAAkAAAAAAAAAAAAAACkAbUaAABMT0cwNV/qs7XthrVfQUnsl63tlaDsi6TsirXs'
    'lYjrgrQubWRQSwECFAMUAAAICAAAACFcDXFNn/UKAACiFwAAHAAAAAAAAAAAAAAApAEXIQAATE9HMDVf6rO17Ya1X+yekeyEseyViOuCtC5tZFBLBQYAAAAA'
    'BQAFAGYBAABGLAAAAAA='
)
_EMBEDDED_ZIP_BYTES = _embedded_b64.b64decode(_EMBEDDED_ZIP_B64)
_zip_path = Path.cwd() / 'LOG05_실습자료.zip'
_zip_path.write_bytes(_EMBEDDED_ZIP_BYTES)
_log05_extract_zip(_zip_path)


## 원본 가상 자료와 작성 양식

아래는 첨부된 **LOG05 실습자료 원본**에서 미션별로 추출한 자료입니다. 모의 기록에서 아직 공개되지 않은 카드나 결과는 미리 확정하지 마세요.

미션 10. 무인 보급장비 투입 전 상태·책임·중지조건 표준 체크리스트 완성

**가. 제공 자료 1 - 지침서 8개 장 구성과 가져올 결과물**

| 장 | 제목 | 포함 내용 | 가져올 결과물 |
| --- | --- | --- | --- |
| 1 | 목적과 적용 범위 | 목적, 적용 장비와 상황, 적용하지 않는 상황, 교전수칙과의 관계 | M1 |
| 2 | 용어 정의 | 계속, 감속, 제자리 안전 정지, 복귀, 원격 회수 요청, 임무 취소 | M1·M3 |
| 3 | 운용 환경과 위험 요인 | 구간 특성, 6개 위험 분류, 사례 | M1 |
| 4 | 투입 전 점검 | 점검 항목표, 신뢰도 G1～G6, 교차 확인 규칙 | M2·M9 |
| 5 | 중지·복귀 조건 | 상태 변화별 처리 단계, 정보 공백, 판단 한계 | M3·M7·M8 |
| 6 | 책임과 승인 | 직책별 책임 한계, 승인 분기점, 사후 감사 | M4·M5 |
| 7 | 관제·훈련·평가 | 관제 상황판, 긴급 중지 명령, 스트레스 훈련, 루브릭, AAR | M6～M8 |
| 8 | 개정 관리 | 연간 개정 주기, 의견 수렴 창구, 책임 직책, 이력 | M8·M9 |

**가. 제공 자료 2 - 모의 심의 역할 카드**

| 역할 | 검토 질문 |
| --- | --- |
| 교리 역할 | 점검 항목과 처리 단계가 일관되는가? 신병도 이해할 용어인가? |
| 법무 역할 | '승인과 중지 결정은 사람이, 교전수칙·지휘계통에 따라'가 드러나는가? 장비 파괴·무력화 쟁점이 검토 연계로 처리되는가? |
| 작전 역할 | 현장에서 20분 안에 점검할 수 있는가? 점검 때문에 긴급 보급이 지나치게 늦어지지 않는가? |

**나. 작성 서식 1 - 근거 표기 점검표(지침서 조항마다 작성)**

| 조항 | 기준 내용 요약 | 근거(미션·가상 자료) | 표기 여부 |
| --- | --- | --- | --- |
| 1-1 |  |  |  |
| 2-1 |  |  |  |
| 3-1 |  |  |  |
| 4-1 |  |  |  |
| 5-1 |  |  |  |
| 6-1 |  |  |  |
| 7-1 |  |  |  |
| 8-1 |  |  |  |

**나. 작성 서식 2 - 모의 심의 지적 사항 반영표**

| 번호 | 역할 | 지적 사항 | 반영 내용 | 반영 위치(장·조) |
| --- | --- | --- | --- | --- |
| 1 |  |  |  |  |
| 2 |  |  |  |  |
| 3 |  |  |  |  |
| 4 |  |  |  |  |
| 5 |  |  |  |  |
| 6 |  |  |  |  |

**나. 작성 서식 3 - 개정 관리: 개정 주기 (   ), 의견 수렴 창구와 방법 (   ), 개정 책임 직책(가상) (   ), 개정 이력 기록 위치 (   )**

**다. 점검표**

| 점검 항목 | 확인 |
| --- | --- |
| 8개 장과 부록을 모두 완성했다 | (   ) |
| 모든 기준 조항에 근거(미션·가상 자료)를 표기했다 | (   ) |
| 교리·법무·작전 역할의 지적 사항을 반영하고 반영 위치를 적었다 | (   ) |
| '승인과 중지 결정은 사람이 하며 실제 교전 판단은 교전수칙·지휘계통에 따른다'는 문구가 1장과 6장에 있다 | (   ) |
| 장비 파괴·무력화 조치를 정의하지 않고 검토 연계로 처리한다는 문구가 5장에 있다 | (   ) |
| 연간 개정 주기와 실무 의견 수렴 창구를 정했다 | (   ) |
| 실제 무인체계 제원·성능을 쓰지 않았다 | (   ) |

## 📋 단계별 수행 안내와 미션 전용 AI 활용

### 활동 1: 8장·부록의 최종 버전을 모은다

**무엇을 하나요?**

#### 1. 단계 개요

- **목표:**
  - 미션 1부터 미션 9까지 수행하는 동안 도출된 모든 결과물, 미확인 영역, 실제/모의/미실시 상태의 데이터를 종합하여, 최종 무인 보급장비 표준 지침서 문서 내 8개 핵심 장(Chapter)과 부록 에 정확하게 연결·편성합니다.
- **수행 방식:**
  - 단편적으로 흩어져 있는 개별 실습 결과들을 하나의 통일된 구조로 집대성하되, 단순한 계획 단계를 넘어 실제 수행된 기록과 대조하여 누락이나 허위 입력이 없도록 엄정하게 정돈합니다.

#### 2. 세부 수행 절차

- **미션 1～9 산출물 및 상태 데이터 통합:**
  - 각 미션별로 생성된 평가 루브릭, AAR 피드백, 7건의 재검증 결과, 1페이지 브리핑 등의 최종 버전을 수집합니다.
- **원본 8장 및 부록 체계에 연결:**
  - 수집된 데이터를 표준 운영 매뉴얼( manual/log05_standard_operational_manual.md )과 양식 파일( LOG05_M10_forms.md , forms/LOG05_M10_forms.md )의 지정된 8개 장과 부록 항목에 각각 매핑합니다.
- **미확인 및 미실시 영역 명시:**
  - 실습 과정에서 정보 공백이나 제약으로 인해 완료되지 못한 미실시 항목이나 미확인 영역을 임의로 채우지 않고, 원본 상태 그대로 정확히 반영합니다.

#### 3. 주의 및 검증 사항

- 계획 단계에 머물렀던 내용을 실제 완료된 것으로 허위 표기하지 않도록 주의해야 합니다.
- 본 단계에서 편성화된 8장과 부록의 초안은 후속 2단계(AI 및 조항별 근거 검토)의 직접적인 검증 대상이 되므로 데이터의 정합성과 출처를 철저히 대조해야 합니다.

#### 직접 물어볼 AI 프롬프트:

- 제공 기록 초안·근거 대조·검산·레드팀·모의 역할 대화

#### 검토:

- 원자료 번호를 표시하고 실제 AI 회신을 수정한 이유를 적으세요.

### 활동 2: AI와 모든 조항의 근거를 검토한다

**무엇을 하나요?**

#### 1. 단계 개요

- **목표:**
  - 표준 체크리스트 및 지침서에 포함된 모든 조항을 대상으로 숫자, G 방향(이동 방향 및 방위각), 조건, 5분 기한, 위임 권한, 명령 상태 등의 핵심 변수들이 상호 모순 없이 정확하게 일치하는지 AI와의 대조를 통해 조항별로 엄밀하게 검토합니다.
- **수행 방식:**
  - 단편적인 육안 검토를 넘어, AI와의 질의응답 및 논리 대조 과정을 통해 각 조항에 명시된 수치와 운용 규칙이 실제 현장 지침 및 이전 미션(M1～M9)의 근거 자료와 완벽하게 부합하는지 교차 검증합니다.

#### 2. 세부 수행 절차

- **핵심 변수 및 조건 조항별 대조 검증:**
  - 숫자 및 기준값: 중지 조건, 장비 적재 중량, 허용 오차 등의 수치가 모든 장(Chapter)에 걸쳐 일관되게 반영되었는지 확인합니다.
  - G 방향 및 방위각: 경로 이탈 및 비상 복귀 시 설정된 이동 방향(G 방향)의 정합성을 검토합니다.
  - 시간 제한(5분 기한) 및 위임 권한: 비상 조치 기한(예: 5분 이내 대응)과 현장 조종수 및 군수참모 간 위임 범위가 명확히 규정되었는지 대조합니다.
  - 명령 상태: 하달된 명령과 모의 회신 로그 간의 상태 정의가 상충되지 않는지 점검합니다.
- **AI 활용 근거 검토 및 수정:**
  - AI 레드팀 대화 및 검토 프롬프트를 활용하여 각 조항의 논리적 허점을 찾아내고, 지적된 항목을 즉각적으로 수정 보완합니다.

#### 3. 주의 및 검증 사항

- 형식적인 검토에 그치지 않고, 조항 간 수치나 조건이 충돌하는 부분이 없는지 원본 데이터와 철저히 대조해야 합니다.
- 본 단계의 검증 결과 및 수정 내역은 양식 파일( forms/LOG05_M10_forms.md , LOG05_M10_forms.md )과 최종 매뉴얼( manual/log05_standar

#### 직접 물어볼 AI 프롬프트:

- 제공 기록 초안·근거 대조·검산·레드팀·모의 역할 대화

#### 검토:

- 원자료 번호를 표시하고 실제 AI 회신을 수정한 이유를 적으세요.

### 활동 3: 모의 심의에 직접 대응한다

**무엇을 하나요?**

#### 1. 단계 개요

- **목표:**
  - 완성된 표준 체크리스트와 지침서 초안을 바탕으로 교리적 타당성, 법무적 적법성, 그리고 작전적 실효성 관점에서 진행되는 가상의 모의 심의(Simulated Review)에 직접 대응하고, 각 심의관들의 지적 사항을 원본 자료와 철저히 대조하여 반영, 보류, 또는 위치 기록 을 명확히 남깁니다.
- **수행 방식:**
  - 형식적인 검토나 일방적인 수용을 지양하고, 실제 군사 작전 및 규정 심의 절차와 동일하게 각 영역별 지적 사항에 대한 방어 논리와 근거를 구축하며, 단순 '승인' 상태와 실제 검토·보완 과정을 엄정하게 구분하여 수행합니다.

#### 2. 세부 수행 절차

- **교리·법무·작전 영역별 지적 사항 대조 분석:**
  - 교리적 심의 대응: 기존 작전 교리와의 정합성을 검토하고, 무인 보급장비 운용 기준이 상위 규정에 저촉되지 않는지 확인합니다.
- **법무적 심의 대응:**
  - 비상 시 자폭/복귀 조건, 책임 한계(군수참모 vs 원격 조종수), 권한 위임 범위 등에 대한 법적 책임과 리스크 요소를 점검합니다.
- **작전적 심의 대응:**
  - 실전 상황에서 부대 작전에 지장을 주지 않고 신속하게 실행 가능한지 운용 타당성을 검증합니다.
  - 심의 결과 조치 결정 (반영·보류·위치 명시): 반영: 지적된 내용이 타당하여 표준 매뉴얼이나 양식 문서의 구체적인 위치( manual/log05_standard_operational_manual.md 등)에 즉시 수정·적용해야 하는 항목.
  - 보류: 추가적인 데이터나 상급 부대 지침 확인이 필요하여 당장 반영하기 어렵고 사유를 남겨야 하는 항목.
- **위치 기록:**
  - 지적 사항과 조치 결과가 매뉴얼 상 어느 장(Chapter) 또는 부록에 반영되었는지 추적 가능하도록 좌표를 명시.
  - 실제 승인과 검토 과정의 엄

#### 직접 물어볼 AI 프롬프트:

- 제공 기록 초안·근거 대조·검산·레드팀·모의 역할 대화

#### 검토:

- 원자료 번호를 표시하고 실제 AI 회신을 수정한 이유를 적으세요.

### 활동 4: 최종본·개정 상태를 제출한다

**무엇을 하나요?**

#### 1. 단계 개요

- **목표:**
  - 앞선 1～3단계를 거쳐 완성된 8개 핵심 장(Chapter)과 부록, 필수 준수 문구, 정기 개정 주기(연 1회), 실무자 의견 수렴 창구, 그리고 실제 검토 대기 상태의 데이터를 최종 점검하여 완벽한 형태의 표준 매뉴얼 및 양식 파일로 확정하고 제출합니다.
- **수행 방식:**
  - 형식적인 저장이나 미완성 상태의 제출을 방지하기 위해 필수 행정 및 운영 요건이 모두 포함되었는지 최종적으로 교차 검증하고, 공식적인 제출 상태로 전환합니다.

#### 2. 세부 수행 절차

- 필수 구성 요소 및 준수 문구 최종 편성화: 8장 및 부록: 미션 1부터 미션 9까지의 모든 분석, 검증, 심의 조치 결과가 반영된 최종 매뉴얼 내용( manual/log05_standard_operational_manual.md ) 정돈.
- 필수 문구 및 개정 체계: 매뉴얼 내에 규정된 필수 지침 문구와 연 1회 정기 개정 원칙을 명시.
- 의견 수렴 창구: 현장 운용 중 발생하는 피드백을 지속적으로 수집하기 위한 의견 창구 및 담당 부서 지정.
- **실제 검토 대기 상태 전환 및 파일 저장:**
  - 작성된 최종 표준 매뉴얼과 양식 파일( LOG05_M10_forms.md , forms/LOG05_M10_forms.md )의 무결성을 확인하고, 검수 및 실무 적용을 위한 '검토 대기' 상태로 저장·제출합니다.

#### 3. 주의 및 검증 사항

- 실제 검토 기록이나 필수 행정 문구(연 1회 개정, 의견 창구 등)가 누락된 채 임의로 완료 처리되지 않도록 세부 항목을 빠짐없이 대조해야 합니다.
- 본 단계를 끝으로 무인 보급장비 표준 체크리스트 완성 프로젝트의 모든 편성이 최종 마무리됩니다.

#### 직접 물어볼 AI 프롬프트:

- 제공 기록 초안·근거 대조·검산·레드팀·모의 역할 대화

#### 검토:

- 원자료 번호를 표시하고 실제 AI 회신을 수정한 이유를 적으세요.

## 제출 전에 확인할 기준

**평가 항목 1:** [정량적 평가] [필수] 표준 지침서 통합 완성 — 표준 지침서 총 8개 장 및 부록을 100% 완성하고 모든 기준 조항에 근거(미션·가상 자료) 출처를 표기했는가, 모의 심의 지적 사항을 반영하고 연간 개정 주기와 실무 의견 수렴 창구를 두었는가, '승인과 중지 결정은 사람이 하며 실제 교전 판단은 교전수칙·지휘계통에 따른다'는 문구를 명시했는가.

- PASS 증거 제공된조건·단위·기간·분모와실제수치/건수/항목근거를연결합니다.
- 목표미달·0기준·미측정은그대로표시합니다.
- 보완 방법 기준별빠진항목과미달원인을찾아수정하고재검증합니다.

**평가 항목 2:** [정성적 평가] Why/How/What: 본인 계획·실제 대화·근거 검토3건 이상·보완·재확인·최종 반영.

- PASS 증거 미션 1～9의 산출물을 통합해 무인 보급장비 투입 전 상태·책임·중지조건 표준 체크리스트(표준 지침서)를 완성하고 모의 심의를 거친다.의판단이제공자료·역할책임·교육용가정과연결됩니다.
- 사실/제안/미확인과실제/모의수행을구분합니다.
- 보완 방법 근거없는원인·규정·완료추정은수정하거나확인필요로보류합니다.

**평가 항목 3:** [진행 추적 및 완료 기준] 실제 조원·두 평가자·2분 판단·20분 문서점검·3분 발표·실제 교관 확인을 AI 모의와 구분한다.

- 미실시를 PASS로 쓰지 않는다.
- 파일: manual/log05_standard_operational_manual.md, LOG05_M10_forms.md, forms/LOG05_M10_forms.md 실제AI요청·답변·근거대조3건이상·보완·재확인·본인최종선택과작성/실행기록이연결됩니다.
- 최종ZIP생성은실제학습완료·교관승인 인증이아닙니다.
- 보완 방법 미작성·미실행·검토누락을보완하고교관검토상태를별도로표시합니다.

**중요:** 정답·실제 측정값·사람의 서명이나 승인을 AI가 만들어 주지 않도록 합니다.

In [ ]:
MISSION_JSON = '{"code": "LOG05_m10", "steps": [{"title": "8장·부록의 최종 버전을 모은다"}, {"title": "AI와 모든 조항의 근거를 검토한다"}, {"title": "모의 심의에 직접 대응한다"}, {"title": "최종본·개정 상태를 제출한다"}]}'
# LOG05 학습 기록 도구 (표준 Python/Jupyter 환경용)
import json, zipfile, io, os, datetime
from pathlib import Path
from IPython.display import display, Markdown
def _load_notebook_widgets():
    from importlib import import_module, invalidate_caches
    import subprocess
    import sys

    try:
        return import_module('ipywidgets')
    except ImportError:
        print('이 노트북 커널에 ipywidgets가 없어 설치합니다.')

    try:
        subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
             '--retries', '0', '--timeout', '10', 'ipywidgets'],
            check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=60,
        )
        invalidate_caches()
        return import_module('ipywidgets')
    except (subprocess.SubprocessError, OSError, ImportError):
        print('이 커널에서 ipywidgets를 설치하거나 불러오지 못했습니다.')
        print('설치 가능한 환경에서는 새 셀에서 %pip install ipywidgets 를 실행한 뒤 이 셀을 다시 실행하세요.')
        print('설치할 수 없는 환경에서는 제공된 Markdown 작성 양식을 사용하세요.')
        return None

widgets = _load_notebook_widgets()
if widgets is not None:
    MISSION_DATA = json.loads(MISSION_JSON)
    CODE = MISSION_DATA['code']
    OUT_DIR = Path(CODE + '_results')
    OUT_DIR.mkdir(exist_ok=True)
    BACKUP = OUT_DIR / (CODE + '_record.json')
    fields = {}
    for no, item in enumerate(MISSION_DATA['steps'], 1):
        fields['activity_%02d' % no] = widgets.Textarea(
            value='', placeholder='내 판단, 확인한 자료 번호, AI 제안과 수정 이유를 직접 작성하세요.',
            description='활동 %s' % no, layout=widgets.Layout(width='98%',height='130px'),
            style={'description_width':'75px'})
    for key, label in [('first_thought','AI에게 묻기 전 내 판단'), ('ai_request','실제로 보낸 AI 요청 전체'),
                       ('ai_response','실제로 받은 AI 답변 전체'), ('source_check','원자료 대조 3건과 수정 이유'),
                       ('final_result','최종 작성 결과와 근거'), ('limits','미실시/미측정/확인 필요 및 교관 검토 상태')]:
        fields[key] = widgets.Textarea(value='', description=label,
           layout=widgets.Layout(width='98%',height='125px'),style={'description_width':'190px'})
    status = widgets.HTML(value='<b>아직 저장하지 않았습니다.</b> 작성과 교관 승인은 별개입니다.')
    restore_input = widgets.Textarea(description='복원 JSON',placeholder='백업 JSON의 내용을 붙여넣으세요.',layout=widgets.Layout(width='98%',height='100px'),style={'description_width':'110px'})

    def payload():
        return {'schema':'log05-student-v1','mission':CODE,'saved_at':datetime.datetime.now(datetime.timezone.utc).isoformat(),
                'answers':{key:w.value for key,w in fields.items()},
                'status':'작성 기록 (학습 PASS 또는 교관 승인 아님)'}

    def save_record(_=None):
        obj=payload()
        if BACKUP.exists():
            stamp=datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f')
            BACKUP.rename(OUT_DIR / (CODE+'_previous_'+stamp+'.json'))
        BACKUP.write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding='utf8')
        status.value='<b>로컬 저장 완료:</b> '+str(BACKUP)+' (PASS 아님)'

    def export_json(_=None):
        save_record()
        raw=BACKUP.read_bytes()
        try:
            from google.colab import files
            files.download(str(BACKUP))
        except ImportError:
            print('Colab이 아닙니다. 저장 위치:',BACKUP.resolve())

    def restore_record(_=None):
        try:
            obj=json.loads(restore_input.value)
            if obj.get('mission')!=CODE or not isinstance(obj.get('answers'),dict):raise ValueError('다른 미션 또는 형식이 맞지 않는 백업입니다.')
            save_record()  # 복원 전 백업 저장
            for key,w in fields.items():w.value=str(obj['answers'].get(key,''))
            status.value='<b>복원 적용:</b> 기존 작성 내용은 이전 JSON으로 보관했습니다. 다시 저장하세요.'
        except Exception as exc:status.value='<b>복원 오류:</b> '+str(exc)

    def export_review_zip(_=None):
        save_record()
        target=OUT_DIR / (CODE+'_review.zip')
        with zipfile.ZipFile(target,'w',compression=zipfile.ZIP_DEFLATED) as zz:
            zz.write(BACKUP,arcname=BACKUP.name)
            text='\n\n'.join('## '+key+'\n'+val for key,val in payload()['answers'].items())
            zz.writestr(CODE+'_work.md',text)
            zz.writestr('README.txt','검토용 보관 파일. 실제 평가 PASS 또는 교관 승인 자료가 아닙니다.')
        try:
            from google.colab import files
            files.download(str(target))
        except ImportError:
            print('Colab이 아닙니다. ZIP 위치:',target.resolve())

    for i,item in enumerate(MISSION_DATA['steps'],1):
        display(Markdown('### 활동 %d. %s' % (i,item['title'])))
        display(fields['activity_%02d' % i])
    for key,label in [('first_thought','최초 판단'),('ai_request','AI 요청 원문'),('ai_response','AI 답변 원문'),
     ('source_check','원자료 대조'),('final_result','최종 결과'),('limits','한계 및 실제 검토 상태')]:
        display(Markdown('### '+label));display(fields[key])
    buttons=[]
    for title,handler in [('작성 내용 저장',save_record),('작성 JSON 내려받기',export_json),('검토용 ZIP 내려받기',export_review_zip),('작성 JSON 복원',restore_record)]:
        b=widgets.Button(description=title,button_style='info');b.on_click(handler);buttons.append(b)
    display(widgets.HBox(buttons[:2]));display(widgets.HBox(buttons[2:]));display(restore_input);display(status)


## 마무리

- 작성 JSON과 검토용 ZIP을 내려받아 보관하세요.
- ZIP은 **검토용**이며 평가 PASS나 교관 승인 증명이 아닙니다.
- 전 단계 결과가 필요하면 본인의 실제 이전 기록을 사용하고, 기록이 없으면 **미실시**로 남기세요.